# Discussion 10: Nonlinear Conjugate Gradient Methods

In this discussion, we will explore
* Nonlinear conjugate gradient methods

---

## Linear Conjugate Gradient Method

Last time we described the **conjugate gradient method** for solving a system of linear equations $Q\mathbf{x}=\mathbf{b}$, which can be interpreted as minimizing a quadratic function, $f(\mathbf{x}) = \frac{1}{2}\mathbf{x}^TQ\mathbf{x} - \mathbf{b}^T\mathbf{x}$. Recall that the conjugate gradient method seeks to mimic the property of Quasi-Newton methods where each new step is $Q$-conjugate to all previous steps, leading to convergence in $n$ steps if $\mathbf{x}\in\mathbb{R}^n$. It does so in the following way:

1. Set $\mathbf{d}_0 = -\nabla f_0$, as in steepest descent.
2. Determine $\alpha_k$ and update $\mathbf{x}_{k+1} = \mathbf{x}_k + \alpha_k\mathbf{d}_k$.
3. Set $\mathbf{d}_{k+1} = -\nabla f_{k+1} + \beta_k\mathbf{d}_k$, where $\beta_k$ is chosen so that $\mathbf{d}_{k+1}$ and $\mathbf{d}_k$ are conjugate in some sense.
4. Repeat 2-3 until convergence.

If $f$ is quadratic, we showed the optimal values for the parameters $\alpha_k$ and $\beta_k$ were given by

$$ \alpha_k = -\frac{\mathbf{d}_k^T\nabla f_k}{\mathbf{d}_k^TQ\mathbf{d}_k},\qquad \beta_k = \frac{\mathbf{d}_k^TQ\nabla f_{k+1}}{\mathbf{d}_k^TQ\mathbf{d}_k} $$

and, after some efficiency considerations, we outlined an efficient version of the linear conjugate gradient method:

1. Set $\mathbf{r}_0= Q\mathbf{x}_0-\mathbf{b}$ (often called the [residual](https://en.wikipedia.org/wiki/Residual_(numerical_analysis))), $\mathbf{d}_0 = -\mathbf{r}_0$.
2. Calculate and store $\mathbf{y}_k=Q\mathbf{d}_k$.
3. Set $\alpha_k = \dfrac{\mathbf{r}_k^T\mathbf{r}_k}{\mathbf{d}_k^T\mathbf{y}_k}$ and update $\mathbf{x}_{k+1} = \mathbf{x}_k + \alpha_k\mathbf{d}_k$.
4. Update $\mathbf{r}_{k+1} = \mathbf{r}_k + \alpha_k\mathbf{y}_k$.
5. Set $\beta_k =\dfrac{\mathbf{r}_{k+1}^T\mathbf{r}_{k+1}}{\mathbf{r}_k^T\mathbf{r}_k}$ and update $\mathbf{d}_{k+1} = -\mathbf{r}_{k+1} + \beta_k\mathbf{d}_k$.
6. Repeat 2-5 until convergence.

This algorithm was shown to indeed converge in at most $n$ steps.

---

## Implementation

Below, we implement CG

In [11]:
import numpy as np
import matplotlib.pyplot as plt

In [12]:
def plot_path(path, func, title, window=[-10,10,-10,10], numContours=50, skip=1):
    '''Plots path defined in (N,2) array "path" on a contour plot of "func" in window "window"'''
    plt.figure(figsize=(10,10))
    X = np.linspace(window[0],window[1],300)
    Y = np.linspace(window[2],window[3],300)
    Xmesh, Ymesh = np.meshgrid(X,Y)
    Z = [f(np.array([x,y])) for x,y in zip(Xmesh.ravel(), Ymesh.ravel())]
    Z = np.array(out).reshape(Xmesh.shape)
    CS = plt.contour(Xmesh, Ymesh, Z, numContours, cmap='jet')
    plt.clabel(CS,inline_spacing=0,fmt='%d')
    plt.axis(window)
    plt.plot(path[0][0], path[0][1], marker='x')
    plt.xlabel('x')
    plt.ylabel('y')
    plt.title(title)

    for i in range(path.shape[0]-1): # iterate through steps
        if i%skip==0:
            # only plot arrows every "skip" iterations
            plt.arrow(path[i,0],path[i,1],path[i+1,0]-path[i,0],path[i+1,1]-path[i,1],
                      color='k',length_includes_head=True)
    plt.show()

In [13]:
def CG_alg_efficient(x0, Q, b, tol=1e-7, max_steps=10000, rho=0.75):
    f = lambda x: 1/2 * x @ Q @ x - b @ x #1/2x^T W x - b^T x
    Df = lambda x: Q @ x - b
    x=x0
    path_CG = [x]
    tol = 1e-7            # stop when gradient is smaller than this amount
    max_steps = 10000     # Maximum number of steps to run the iteration
    rho = 0.75            # rho for backtracking
    i=0                   # iteration count
    gk = Df(x)          # current gradient
    dk = -gk
    dks = [dk]
    gks = [gk]

    while np.linalg.norm(gk)>tol and i<max_steps:   
        r0 = Q * x - b
        alpha = - dk @ gk / (dk @ Q @ dk)
        xnew = x + alpha*dk
        gk1 = Df(xnew)
        beta = (dk @ Q @ gk1)/ (dk @ Q @ dk)
        dk = -gk1+ beta*dk
        dks.append(dk)
        path_CG.append(xnew)
        x = xnew
        gk = gk1
        gks.append(gk)
        
        i += 1

    path_CG=np.array(path_CG)
    dks = np.array(dks)
    gks = np.array(gks)
    print(f'After {i} iterations, approximate minimum is {f(x)} at {x}')
    return path_CG, dks, gks

In [14]:
def CG_alg(x0, Q, b, tol=1e-7, max_steps=10000, rho=0.75):
    f = lambda x: 1/2 * x @ Q @ x - b @ x
    Df = lambda x: Q @ x - b
    x=x0
    path_CG = [x]
    tol = 1e-7            # stop when gradient is smaller than this amount
    max_steps = 10000     # Maximum number of steps to run the iteration
    rho = 0.75            # rho for backtracking
    i=0                   # iteration count
    gk = Df(x)          # current gradient
    dk = -gk
    dks = [dk]
    gks = [gk]

    while np.linalg.norm(gk)>tol and i<max_steps:    
        alpha = - dk @ gk / (dk @ Q @ dk)
        xnew = x + alpha*dk
        gk1 = Df(xnew)
        beta = (dk @ Q @ gk1)/ (dk @ Q @ dk)
        dk = -gk1+ beta*dk
        dks.append(dk)
        path_CG.append(xnew)
        x = xnew
        gk = gk1
        gks.append(gk)
        
        i += 1

    path_CG=np.array(path_CG)
    dks = np.array(dks)
    gks = np.array(gks)
    print(f'After {i} iterations, approximate minimum is {f(x)} at {x}')
    return path_CG, dks, gks

In [15]:
Q = np.array([[9,0],[0,4]])
b = np.array([2,4])
x0 = (1.5,2)
path_CG, dks, gks = CG_alg(x0, Q, b)
print(path_CG)

After 2 iterations, approximate minimum is -2.2222222222222228 at [0.22222222 1.        ]
[[1.5        2.        ]
 [0.14072155 1.52720749]
 [0.22222222 1.        ]]


In [16]:
f = lambda x: 1/2*x@Q@x - b@x
plot_path(path_CG,f,'CG method')

NameError: name 'out' is not defined

<Figure size 720x720 with 0 Axes>

### Verifying Some properties:

Conjugacy: $d_i^T Q d_j = 0$

In [219]:
for i in range(len(dks)):
    for j in range(i+1, len(dks)):
        print(dks[i]@Q@dks[j])

1.4210854715202004e-14
-6.894484982922222e-14
0.0


Orthogonality: $g_{k+1}^T d_j = 0$

In [220]:
for i in range(1,len(gks)):
    for j in range(0,i):
        print("i={0}, j={1} dot product {2:.2f}".format(i,j,gks[i]@dks[j]))

i=1, j=0 dot product 0.00
i=2, j=0 dot product 0.00
i=2, j=1 dot product -0.00
